# Load Data

In [18]:
import pickle
from pathlib import Path

import numpy as np

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

In [55]:
from sklearn.model_selection import cross_val_score

In [29]:
n_gram_feats = np.load("./out/verse_n_gram_array.npy")
chargram_feats = np.load("./out/chargram_array.npy")
labels = np.load("./out/labels.npy")

In [23]:
ot_verse_nums = pickle.load(Path("./out/ot_verse_nums.pickle").open("br"))

nt_verse_nums = pickle.load(Path("./out/nt_verse_nums.pickle").open("br"))

## Training the algorithm

In [24]:
ot_verse_nums

[('Genesis', 1533), ('Exodus', 582), ('Deuteronomy', 555)]

In [25]:
nt_verse_nums

[('Matthew', 1071),
 ('Mark', 678),
 ('Luke', 1098),
 ('John', 879),
 ('Acts', 1007)]

In [ ]:
n_gram_train, n_gram_test, n_gram_train_y, n_gram_test_y = train_test_split(
    n_gram_feats, labels, test_size=0.2, random_state=0
)

gnb = GaussianNB()

clf = gnb.fit(n_gram_train, n_gram_train_y)
y_pred = clf.predict(n_gram_test)

print(
    "Number of mislabeled points out of a total %d points : %d"
    % (n_gram_test.shape[0], (n_gram_test_y != y_pred).sum())
)

In [58]:
fold = 5
for i in range(30):
    print(f"random_state={i}")
    chargram_X_train, chargram_X_test, chargram_y_train, chargram_y_test = (
        train_test_split(chargram_feats, labels, test_size=0.2, random_state=i)
    )

    gnb = GaussianNB()

    c_clf = gnb.fit(chargram_X_train, chargram_y_train)
    y_pred = c_clf.predict(chargram_X_test)

    print(
        "Number of mislabeled points out of a total %d points : %d"
        % (chargram_X_test.shape[0], (chargram_y_test != y_pred).sum())
    )
    cv_fscores = cross_val_score(
        gnb, chargram_X_train, chargram_y_train, cv=fold, scoring="f1_macro"
    )
    print(
        "%0.6f F1 score with a standard deviation of %0.6f"
        % (cv_fscores.mean(), cv_fscores.std())
    )
    print(f"{fold}-fold cross-validation f scores: {cv_fscores}")

random_state=0
Number of mislabeled points out of a total 1481 points : 57
0.962554 F1 score with a standard deviation of 0.002242
5-fold cross-validation f scores: [0.96167411 0.95960194 0.96423146 0.96592342 0.96133896]
random_state=1
Number of mislabeled points out of a total 1481 points : 50
0.958637 F1 score with a standard deviation of 0.004766
5-fold cross-validation f scores: [0.95883273 0.9545977  0.95271275 0.96620355 0.96083978]
random_state=2
Number of mislabeled points out of a total 1481 points : 59
0.961249 F1 score with a standard deviation of 0.004692
5-fold cross-validation f scores: [0.96421129 0.96066238 0.95595443 0.95684967 0.96856847]
random_state=3
Number of mislabeled points out of a total 1481 points : 51
0.961010 F1 score with a standard deviation of 0.007810
5-fold cross-validation f scores: [0.97164032 0.96618571 0.94939915 0.96219924 0.95562607]
random_state=4
Number of mislabeled points out of a total 1481 points : 61
0.961638 F1 score with a standard dev

In [60]:
fold = 5
chargram_X_train, chargram_X_test, chargram_y_train, chargram_y_test = (
    train_test_split(chargram_feats, labels, test_size=0.2, random_state=0)
)

gnb = GaussianNB()

c_clf = gnb.fit(chargram_X_train, chargram_y_train)
y_pred = c_clf.predict(chargram_X_test)

print(
    "Number of mislabeled points out of a total %d points : %d"
    % (chargram_X_test.shape[0], (chargram_y_test != y_pred).sum())
)
cv_fscores = cross_val_score(
    gnb, chargram_X_train, chargram_y_train, cv=fold, scoring="f1_macro"
)
print(
    "%0.6f F1 score with a standard deviation of %0.6f"
    % (cv_fscores.mean(), cv_fscores.std())
)
print(f"{fold}-fold cross-validation f scores: {cv_fscores}")

Number of mislabeled points out of a total 1481 points : 57
0.962554 F1 score with a standard deviation of 0.002242
5-fold cross-validation f scores: [0.96167411 0.95960194 0.96423146 0.96592342 0.96133896]


In [65]:
import json

In [66]:
# pos = sum([book[1] for book in ot_verse_nums])
# print(pos)
pos = 0

ot_mislabels = {
    "Genesis": [],
    "Exodus": [],
    "Deuteronomy": [],
}

for book in ot_verse_nums:
    print(book[0])
    book_verses = []
    verse_labels = []
    for i in range(book[1]):
        book_verses.append(verse_chargram_feat[pos])
        verse_labels.append(0)
        pos += 1

    book_verses = np.array(book_verses)
    verse_labels = np.array(verse_labels)
    book_pred = c_clf.predict(book_verses)
    if book[0] == "Genesis":
        print(book_pred)
    mislabeled_points = [i for i in range(len(book_pred)) if book_pred[i] == 1]
    print(
        "Number of mislabeled points out of a total %d points : %d"
        % (book_verses.shape[0], (verse_labels != book_pred).sum())
    )

with Path(f"./out/prediction_data_{book[0]}.json").open(mode="w") as fp:
    json.dump(mislabels, fp)

Genesis
[1 0 0 ... 0 0 0]
Number of mislabeled points out of a total 1533 points : 46
Exodus
Number of mislabeled points out of a total 582 points : 20
Deuteronomy
Number of mislabeled points out of a total 555 points : 55


### Quick evaluations

In [ ]:
fold = 5

print(f"Gaussian Naïve Bayes with Word n-grams, {fold}-fold cross-validation:")

# Calculate Accuracy through Cross-Validation
cv_scores = cross_val_score(
    gnb, n_gram_train, n_gram_train_y, cv=fold
)  # cross-validation scores
print(
    "%0.6f accuracy with a standard deviation of %0.6f"
    % (cv_scores.mean(), cv_scores.std())
)

# Calculate F1 score through Cross-Validation
cv_fscores = cross_val_score(
    gnb, n_gram_train, n_gram_train_y, cv=fold, scoring="f1_macro"
)
print(
    "%0.6f F1 score with a standard deviation of %0.6f"
    % (cv_fscores.mean(), cv_fscores.std())
)

In [ ]:
fold = 5

print(
    f"Gaussian Naïve Bayes with Character n-grams, {fold}-fold cross-validation:"
)

# Calculate Accuracy through Cross-Validation
cv_scores = cross_val_score(
    gnb, chargram_X_train, chargram_y_train, cv=fold
)  # cross-validation scores
print(
    "%0.6f accuracy with a standard deviation of %0.6f"
    % (cv_scores.mean(), cv_scores.std())
)

# Calculate F1 score through Cross-Validation
cv_fscores = cross_val_score(
    gnb, chargram_X_train, chargram_y_train, cv=fold, scoring="f1_macro"
)
print(
    "%0.6f F1 score with a standard deviation of %0.6f"
    % (cv_fscores.mean(), cv_fscores.std())
)